# Session 1: LangChain Fundamentals (60 minutes)

## 🎯 Learning Objectives
- Understand why LangChain exists
- Work with LLMs and ChatModels
- Create and use Prompt Templates
- Parse structured outputs
- Master LCEL (LangChain Expression Language)

## 📋 Problem Statement
We're building an AI Research Assistant. This session covers the foundation: basic LLM interactions.

## ⏱️ Session Breakdown
- 10 min: Why LangChain? The problem with raw APIs
- 25 min: Core Components (Models, Prompts, Parsers)
- 15 min: Hands-on Exercise
- 10 min: Recap & Q&A

---

## 🖥️ Using Ollama (Local, FREE, No API Limits!)
We're using **TinyLlama** via Ollama - runs locally on minimal hardware (even college lab PCs!)

## 1. Setup - Run this first!

In [ ]:
# Install required packages (run once)
# !pip install langchain langchain-ollama python-dotenv ipywidgets

# ============================================
# 🚀 OLLAMA SETUP (One-time, takes 2 minutes)
# ============================================
# 
# 1. Download Ollama: https://ollama.ai/download
#    - Windows: Download and run installer
#    - Mac: brew install ollama
#    - Linux: curl -fsSL https://ollama.ai/install.sh | sh
#
# 2. Start Ollama (open terminal):
#    > ollama serve
#
# 3. Pull TinyLlama (very small, ~600MB, runs on any PC!):
#    > ollama pull tinyllama
#
# ============================================

print("📥 Make sure you've installed Ollama and pulled tinyllama!")
print("   Run in terminal: ollama pull tinyllama")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Disable LangSmith tracing by default (optional feature - covered in Session 4)
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# Check if Ollama is running
import subprocess
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
    if "tinyllama" in result.stdout.lower():
        print("✅ Ollama is running and TinyLlama is available!")
    else:
        print("⚠️  TinyLlama not found. Run: ollama pull tinyllama")
except:
    print("⚠️  Ollama not detected. Please:")
    print("   1. Install from: https://ollama.ai/download")  
    print("   2. Run: ollama serve")
    print("   3. Run: ollama pull tinyllama")

### 💡 Why TinyLlama?
- **Tiny**: Only 1.1B parameters (~600MB download)
- **Fast**: Runs on CPU, no GPU needed!
- **Free**: No API keys, no rate limits
- **Private**: All data stays on your computer
- **Perfect for learning**: Same LangChain code works with any model

In [ ]:
# List available Ollama models on your system
import subprocess
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    print("📋 Available Ollama models:")
    print(result.stdout)
    
    # Alternative small models you can try:
    print("\n💡 Other small models (if tinyllama doesn't work):")
    print("   ollama pull qwen2:0.5b    # Smallest! Only 400MB")
    print("   ollama pull phi3:mini     # Microsoft's small model")  
    print("   ollama pull gemma2:2b     # Google's 2B model")
except Exception as e:
    print(f"⚠️  Could not list models: {e}")
    print("   Make sure Ollama is running: ollama serve")

## 2. The Problem: Raw API Calls Are Verbose

❌ **WITHOUT LangChain** - Direct model calls are verbose and provider-specific:

In [ ]:
# This is how you'd typically call a model without any framework:
"""
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "tinyllama",
        "prompt": "What is artificial intelligence?",
        "stream": False
    }
)
print(response.json()["response"])
"""

# Problems with this approach:
# 1. Verbose HTTP setup for every call
# 2. Different interfaces per provider
# 3. No standardized message format
# 4. Hard to switch between local/cloud models
# 5. No built-in prompt management

print("Problems with raw API/HTTP calls:")
print("1. Verbose setup for every call")
print("2. Different interface per provider")
print("3. No standardized message format")
print("4. Hard to switch between local/cloud")
print("5. No built-in prompt management")

## 3. The Solution: LangChain's Unified Interface

✅ **WITH LangChain** - Clean and Provider-Agnostic

In [ ]:
from langchain_ollama import ChatOllama

# Initialize the model - Qwen 0.5B via Ollama (LOCAL, FREE, SUPER TINY!)
llm = ChatOllama(
    model="qwen2:0.5b",        # Qwen 0.5B - only 0.5B params, ~400MB, runs on ANY computer!
    temperature=0.7,          # Creativity: 0 = deterministic, 1 = creative
)

# Simple invocation - same interface works for ANY provider!
response = llm.invoke("What is artificial intelligence in one sentence?")
print("🤖 Response:", response.content)
print("📊 Type:", type(response))

## 4. Understanding the Response

LangChain returns message objects, not raw strings. This is more structured and useful!

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# The response is an AIMessage object
print("--- Inspecting the Response ---")
print(f"Content: {response.content}")
print(f"Type: {type(response).__name__}")

# You can also access metadata
if hasattr(response, 'response_metadata'):
    print(f"Metadata: {response.response_metadata}")

## 5. Different Message Types

LangChain uses typed messages for better structure:
- **SystemMessage**: Instructions for the AI's behavior
- **HumanMessage**: User's input
- **AIMessage**: AI's response

In [ ]:
# Using message objects for more control
messages = [
    SystemMessage(content="You are a research assistant specializing in technology."),
    HumanMessage(content="Explain machine learning in simple terms.")
]

response = llm.invoke(messages)
print("🤖 Response:", response.content)

## 6. Prompt Templates - Reusable Prompts

🎯 **Problem**: Hardcoded prompts are hard to maintain and reuse.

💡 **Solution**: Prompt Templates with variables!

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Simple template with variables
simple_template = PromptTemplate(
    template="Explain {topic} in {style} terms.",
    input_variables=["topic", "style"]
)
# Format the template
formatted = simple_template.format(topic="neural networks", style="simple")
print("📝 Formatted prompt:", formatted)

## 7. Chat Prompt Templates

For chat models, use `ChatPromptTemplate` with message roles.

In [ ]:
# More sophisticated template for our Research Assistant
research_template = ChatPromptTemplate.from_messages([
    ("system", """You are an expert research assistant. Your role is to:
    - Provide accurate, well-researched information
    - Cite sources when possible
    - Explain complex topics clearly
    - Admit when you don't know something"""),
    ("human", "Research topic: {topic}\n\nProvide a brief overview.")
])

# Create the prompt
prompt = research_template.format_messages(topic="Quantum Computing")
print("📝 Formatted messages:")
for msg in prompt:
    print(f"  [{msg.type}]: {msg.content[:50]}...")

## 8. LCEL - LangChain Expression Language

🎯 **LCEL**: The modern way to compose LangChain components

The pipe operator (`|`) connects components:
```python
prompt | llm | parser
```

This creates a "chain" that flows data through each component.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Create components
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain {topic} in 2-3 sentences.")
])

output_parser = StrOutputParser()  # Converts AIMessage to string

# LCEL Chain: Connect with pipe operator
chain = prompt | llm | output_parser

# Invoke the chain
result = chain.invoke({"topic": "blockchain"})
print("🔗 Chain result:", result)
print("📊 Type:", type(result))  # Now it's a string!

## 9. Output Parsers - Structured Output

🎯 **Problem**: LLMs return text. We often need structured data.

💡 **Solution**: Output Parsers convert text to structured formats.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import List

# ⚠️ IMPORTANT: Small models (0.5B) struggle with strict JSON parsing
# This cell demonstrates OUTPUT PARSING concepts - results may vary with tiny models

# Define the structure we WANT (not guaranteed with very small models)
class ResearchSummary(BaseModel):
    """Structure for research summaries"""
    topic: str = Field(description="The research topic")
    summary: str = Field(description="Brief summary in 1-2 sentences")

# For small models, use a simpler approach - parse the raw text output
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research assistant. Keep responses SHORT and SIMPLE.
    Format your answer like this:
    Topic: [topic name]
    Summary: [1-2 sentences]"""),
    ("human", "Research topic: {topic}")
])

# Use simple string parser instead of complex JSON
simple_chain = simple_prompt | llm | StrOutputParser()

# Get result
result = simple_chain.invoke({"topic": "Artificial Intelligence"})
print("📝 Simple Parsed Result:")
print(result)

print("\n✅ For production code with reliable structured output:")
print("   - Use larger models (7B+)")
print("   - Use models specifically trained for JSON (openhermes, etc.)")
print("   - Or implement custom parsing logic")

## 10. Batch Processing

Process multiple inputs efficiently with `.batch()`

In [ ]:
# Create a simple chain
simple_chain = (
    ChatPromptTemplate.from_template("What is the capital of {country}?")
    | llm
    | StrOutputParser()
)

# Batch process multiple countries
countries = [
    {"country": "France"},
    {"country": "Japan"},
    {"country": "Brazil"}
]

results = simple_chain.batch(countries)
print("🌍 Batch Results:")
for country, result in zip(countries, results):
    print(f"  {country['country']}: {result}")

## 11. Streaming Output

Stream responses for better UX (shows tokens as they're generated)

In [ ]:
print("📡 Streaming output:")
streaming_chain = (
    ChatPromptTemplate.from_template("Write a haiku about {topic}")
    | llm
    | StrOutputParser()
)

# Stream the response
for chunk in streaming_chain.stream({"topic": "programming"}):
    print(chunk, end="", flush=True)
print()  # New line at end

## 12. 🏋️ Exercise: Build Your Research Assistant Prompt

**Task**: Create a research assistant chain that:
1. Takes a topic and depth level (brief/detailed)
2. Returns structured output
3. Uses proper prompt engineering

Try modifying the template below!

In [ ]:
# TODO: Create your research assistant
exercise_template = ChatPromptTemplate.from_messages([
    ("system", """You are an expert research assistant.
    Depth level: {depth}
    - If brief: Provide 2-3 sentences
    - If detailed: Provide a comprehensive overview with sections"""),
    ("human", "Research topic: {topic}")
])

# Create the chain
exercise_chain = exercise_template | llm | StrOutputParser()

# Test it!
result = exercise_chain.invoke({"topic": "Renewable Energy", "depth": "brief"})
print(result)

## 13. Alternative Models (Same Interface!)

LangChain works with many providers. Here's how to switch:

In [ ]:
# TinyLlama via Ollama (what we're using - LOCAL & FREE!)
from langchain_ollama import ChatOllama
tiny_llm = ChatOllama(model="tinyllama")

# Other small Ollama models:
# qwen_llm = ChatOllama(model="qwen2:0.5b")   # Even smaller! 0.5B params
# phi_llm = ChatOllama(model="phi3:mini")     # Microsoft's efficient model
# gemma_llm = ChatOllama(model="gemma2:2b")   # Google's 2B model

# Cloud options (require API keys):
# from langchain_google_genai import ChatGoogleGenerativeAI
# gemini_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# from langchain_openai import ChatOpenAI
# openai_llm = ChatOpenAI(model="gpt-4o-mini")

print("✅ LangChain supports 50+ model providers!")
print("🖥️  We're using TinyLlama via Ollama - LOCAL, FREE, NO LIMITS!")

## 📚 Session 1 Recap

### Key Takeaways:

1. **LangChain provides a UNIFIED INTERFACE for LLMs**
   - Same code works with Google, OpenAI, Anthropic, etc.
   - Easy to switch providers

2. **Core Components:**
   - `ChatModels`: The LLM wrapper (ChatGoogleGenerativeAI, etc.)
   - `Prompt Templates`: Reusable, parameterized prompts
   - `Output Parsers`: Convert text to structured data

3. **LCEL (LangChain Expression Language):**
   - Use `|` pipe operator to chain components
   - `prompt | llm | parser`

4. **Key Methods:**
   - `.invoke()` - Single input
   - `.batch()` - Multiple inputs
   - `.stream()` - Streaming output

---

### 🔜 Next Session: Chains & Memory
- Building multi-step workflows
- Making the LLM remember conversation context

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 1 COMPLETE! 🎉                                  ║
║                                                                            ║
║  Next: Session 2 - Chains & Memory                                         ║
║  File: 02_chains_and_memory.ipynb                                          ║
║                                                                            ║
║  "Now we can call LLMs easily. But what if we need multiple steps?         ║
║   What if the LLM forgets our conversation? That's next!"                  ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")